# Data Loading & Basic Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

df = pd.read_csv(url, names=columns)

print(df.head())
df.shape

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


(768, 9)

In [ ]:
# Separate features and labels
X = df.iloc[:,:-1]
y= df.iloc[:,-1]

# Train test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# Scaling


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Numpy to Tensor Conversion
PyTorch works with TENSORS.
Not numpy arrays.
So we convert.

In [ ]:
# Convert to tensors
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()

y_train_tensor = torch.from_numpy(y_train.values).float().view(-1,1)
y_test_tensor = torch.from_numpy(y_test.values).float().view(-1,1)


# Defining the Model


In [ ]:
class OurNN():

  def __init__(self, input_size):
    #intialize wight and bias
    self.weights= torch.rand(input_size, 1, requires_grad=True)
    self.bias= torch.zeros(1, requires_grad=True)

  def forward(self, X):
    #forward pass
    z= torch.matmul(X, self.weights) + self.bias
    return torch.sigmoid(z)

  def loss_function(self,y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)

    loss = -(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred))
    return loss.mean()


# Training Pipeline


In [ ]:
#hyperparameters

learning_rate = 0.01
epochs = 50


# creating model instance
model = OurNN(X_train_tensor.shape[1])

#training loop

for epoch in range(epochs):

    # 1. Forward pass
    y_pred = model.forward(X_train_tensor)

    # 2. Loss calculation
    loss = model.loss_function(y_pred, y_train_tensor)

    # 3. Backward pass
    loss.backward()


    # 4. Update parameters
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # 5. Zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    print(f"Epoch : {epoch+1} , Loss : {loss.item()}")


Epoch : 1 , Loss : 0.6434996724128723
Epoch : 2 , Loss : 0.6429520845413208
Epoch : 3 , Loss : 0.6424055099487305
Epoch : 4 , Loss : 0.64186030626297
Epoch : 5 , Loss : 0.6413165926933289
Epoch : 6 , Loss : 0.6407740116119385
Epoch : 7 , Loss : 0.6402326226234436
Epoch : 8 , Loss : 0.6396927237510681
Epoch : 9 , Loss : 0.6391539573669434
Epoch : 10 , Loss : 0.638616681098938
Epoch : 11 , Loss : 0.6380802392959595
Epoch : 12 , Loss : 0.6375453472137451
Epoch : 13 , Loss : 0.637011706829071
Epoch : 14 , Loss : 0.6364791989326477
Epoch : 15 , Loss : 0.635948121547699
Epoch : 16 , Loss : 0.6354182958602905
Epoch : 17 , Loss : 0.6348896026611328
Epoch : 18 , Loss : 0.6343623399734497
Epoch : 19 , Loss : 0.6338362097740173
Epoch : 20 , Loss : 0.63331139087677
Epoch : 21 , Loss : 0.6327879428863525
Epoch : 22 , Loss : 0.6322656273841858
Epoch : 23 , Loss : 0.6317445635795593
Epoch : 24 , Loss : 0.6312249302864075
Epoch : 25 , Loss : 0.6307063698768616
Epoch : 26 , Loss : 0.630189061164856
Epo

# Model Evaluation

In [ ]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred_class = (y_pred>0.5 ).float()
    accuracy = (y_pred_class == y_test_tensor).float().mean()

    print(f"Accuracy : {accuracy.item()}")

Accuracy : 0.701298713684082
